In [1]:
from pathlib import Path

BASE_DIR = Path("../data/synthetic")

PRIMARY_KEYS = {
    "customers_master": "customer_id",
    "products_master": "product_id",
    "stores_master": "store_id",
    "salespersons_master": "salesperson_id",
    "promotions_master": "promotion_id",
    "sales": "sales_line_id",
    "returns": "return_id",
    "targets": "target_id"
}

print("=" * 70)
print("02.29 - PRIMARY KEY VALIDATION")
print("=" * 70)

print("\nPrimary keys to validate:\n")

for dataset, key in PRIMARY_KEYS.items():
    print(f"{dataset:<25} -> {key}")

print("\n02.29 Step 1 preparation: PASSED")

02.29 - PRIMARY KEY VALIDATION

Primary keys to validate:

customers_master          -> customer_id
products_master           -> product_id
stores_master             -> store_id
salespersons_master       -> salesperson_id
promotions_master         -> promotion_id
sales                     -> sales_line_id
returns                   -> return_id
targets                   -> target_id

02.29 Step 1 preparation: PASSED


In [2]:
import pandas as pd

print("=" * 70)
print("02.29 - STEP 2 - PRIMARY KEY COMPLETENESS & UNIQUENESS")
print("=" * 70)

validation_results = []

for dataset, key_column in PRIMARY_KEYS.items():

    # Handle monthly transaction files separately
    if dataset == "sales":
        files = sorted(BASE_DIR.glob("sales_*.csv"))
    elif dataset == "returns":
        files = sorted(BASE_DIR.glob("returns_*.csv"))
    elif dataset == "targets":
        files = [BASE_DIR / "targets.csv"]
    else:
        files = [BASE_DIR / f"{dataset}.csv"]

    total_rows = 0
    missing_keys = 0
    blank_keys = 0
    duplicate_keys = 0

    for file_path in files:

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        total_rows += len(df)

        missing_keys += df[key_column].isna().sum()

        blank_keys += (
            df[key_column]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        duplicate_keys += df[key_column].duplicated().sum()

    valid = (
        missing_keys == 0
        and blank_keys == 0
        and duplicate_keys == 0
    )

    validation_results.append(valid)

    print(f"\n{dataset}")
    print(f"Rows            : {total_rows:,}")
    print(f"Primary key     : {key_column}")
    print(f"Missing keys    : {missing_keys:,}")
    print(f"Blank keys      : {blank_keys:,}")
    print(f"Duplicate keys  : {duplicate_keys:,}")
    print(f"Status          : {'VALID' if valid else 'NEEDS REVIEW'}")


print("\n" + "=" * 70)

if all(validation_results):
    print("02.29 Step 2 validation: PASSED")
else:
    print("02.29 Step 2 validation: NEEDS REVIEW")

02.29 - STEP 2 - PRIMARY KEY COMPLETENESS & UNIQUENESS

customers_master
Rows            : 10,000
Primary key     : customer_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

products_master
Rows            : 2,500
Primary key     : product_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

stores_master
Rows            : 60
Primary key     : store_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

salespersons_master
Rows            : 500
Primary key     : salesperson_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

promotions_master
Rows            : 72
Primary key     : promotion_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

sales
Rows            : 2,118,200
Primary key     : sales_line_id
Missing keys    : 0
Blank keys      : 0
Duplicate keys  : 0
Status          : VALID

returns
Rows        

In [4]:
import pandas as pd

print("=" * 70)
print("02.29 - STEP 3 - PRIMARY KEY DATA TYPE VALIDATION")
print("=" * 70)

all_valid = True

for dataset, key_column in PRIMARY_KEYS.items():

    if dataset == "sales":
        files = sorted(BASE_DIR.glob("sales_*.csv"))
    elif dataset == "returns":
        files = sorted(BASE_DIR.glob("returns_*.csv"))
    elif dataset == "targets":
        files = [BASE_DIR / "targets.csv"]
    else:
        files = [BASE_DIR / f"{dataset}.csv"]

    for file_path in files:

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        dtype = df[key_column].dtype

        # Accept pandas string/object types as valid identifier types
        is_valid = pd.api.types.is_string_dtype(df[key_column])

        if not is_valid:
            all_valid = False

        print(
            f"{file_path.name:<35} | "
            f"{key_column:<20} | "
            f"dtype: {str(dtype):<10} | "
            f"{'VALID' if is_valid else 'NEEDS REVIEW'}"
        )

print("\n" + "=" * 70)

if all_valid:
    print("02.29 Step 3 validation: PASSED")
else:
    print("02.29 Step 3 validation: NEEDS REVIEW")

02.29 - STEP 3 - PRIMARY KEY DATA TYPE VALIDATION
customers_master.csv                | customer_id          | dtype: str        | VALID
products_master.csv                 | product_id           | dtype: str        | VALID
stores_master.csv                   | store_id             | dtype: str        | VALID
salespersons_master.csv             | salesperson_id       | dtype: str        | VALID
promotions_master.csv               | promotion_id         | dtype: str        | VALID
sales_2024-01.csv                   | sales_line_id        | dtype: str        | VALID
sales_2024-02.csv                   | sales_line_id        | dtype: str        | VALID
sales_2024-03.csv                   | sales_line_id        | dtype: str        | VALID
sales_2024-04.csv                   | sales_line_id        | dtype: str        | VALID
sales_2024-05.csv                   | sales_line_id        | dtype: str        | VALID
sales_2024-06.csv                   | sales_line_id        | dtype: str        |

In [5]:
print("=" * 70)
print("02.29 - STEP 4 - GLOBAL PRIMARY KEY UNIQUENESS")
print("=" * 70)

all_valid = True

# Sales primary keys across all monthly files
sales_files = sorted(BASE_DIR.glob("sales_*.csv"))

sales_keys = []

for file_path in sales_files:
    df = pd.read_csv(file_path, usecols=["sales_line_id"])
    sales_keys.extend(df["sales_line_id"].dropna().astype(str).tolist())

sales_duplicate_count = len(sales_keys) - len(set(sales_keys))

print("\nSALES")
print(f"Files checked        : {len(sales_files)}")
print(f"Total primary keys   : {len(sales_keys):,}")
print(f"Duplicate keys       : {sales_duplicate_count:,}")

if sales_duplicate_count != 0:
    all_valid = False


# Returns primary keys across all monthly files
returns_files = sorted(BASE_DIR.glob("returns_*.csv"))

return_keys = []

for file_path in returns_files:
    df = pd.read_csv(file_path, usecols=["return_id"])
    return_keys.extend(df["return_id"].dropna().astype(str).tolist())

return_duplicate_count = len(return_keys) - len(set(return_keys))

print("\nRETURNS")
print(f"Files checked        : {len(returns_files)}")
print(f"Total primary keys   : {len(return_keys):,}")
print(f"Duplicate keys       : {return_duplicate_count:,}")

if return_duplicate_count != 0:
    all_valid = False


# Target primary keys
target_file = BASE_DIR / "targets.csv"

if target_file.exists():

    targets_df = pd.read_csv(target_file, usecols=["target_id"])

    target_keys = targets_df["target_id"].dropna().astype(str).tolist()

    target_duplicate_count = (
        len(target_keys) - len(set(target_keys))
    )

    print("\nTARGETS")
    print(f"Total primary keys   : {len(target_keys):,}")
    print(f"Duplicate keys       : {target_duplicate_count:,}")

    if target_duplicate_count != 0:
        all_valid = False


print("\n" + "=" * 70)

if all_valid:
    print("02.29 Step 4 validation: PASSED")
else:
    print("02.29 Step 4 validation: NEEDS REVIEW")

02.29 - STEP 4 - GLOBAL PRIMARY KEY UNIQUENESS

SALES
Files checked        : 32
Total primary keys   : 2,118,200
Duplicate keys       : 0

RETURNS
Files checked        : 32
Total primary keys   : 118,400
Duplicate keys       : 0

TARGETS
Total primary keys   : 8,980
Duplicate keys       : 0

02.29 Step 4 validation: PASSED


In [6]:
print("=" * 70)
print("02.29 - STEP 5 - FINAL PRIMARY KEY VALIDATION")
print("=" * 70)

datasets_to_check = {
    "customers_master": ("customers_master.csv", "customer_id"),
    "products_master": ("products_master.csv", "product_id"),
    "stores_master": ("stores_master.csv", "store_id"),
    "salespersons_master": ("salespersons_master.csv", "salesperson_id"),
    "promotions_master": ("promotions_master.csv", "promotion_id"),
    "targets": ("targets.csv", "target_id"),
}

all_valid = True

for dataset, (filename, key_column) in datasets_to_check.items():

    file_path = BASE_DIR / filename

    df = pd.read_csv(file_path, dtype={key_column: str})

    missing = df[key_column].isna().sum()
    blank = df[key_column].astype(str).str.strip().eq("").sum()
    duplicates = df[key_column].duplicated().sum()

    valid = (
        missing == 0
        and blank == 0
        and duplicates == 0
    )

    if not valid:
        all_valid = False

    print(f"\n{dataset}")
    print(f"Rows             : {len(df):,}")
    print(f"Primary key      : {key_column}")
    print(f"Missing keys     : {missing:,}")
    print(f"Blank keys       : {blank:,}")
    print(f"Duplicate keys   : {duplicates:,}")
    print(f"Status            : {'VALID' if valid else 'NEEDS REVIEW'}")


# Sales and Returns were already validated globally in Step 4.
print("\nsales")
print("Primary key      : sales_line_id")
print("Global duplicates: 0")
print("Status            : VALID")

print("\nreturns")
print("Primary key      : return_id")
print("Global duplicates: 0")
print("Status            : VALID")


print("\n" + "=" * 70)

if all_valid:
    print("02.29 Step 5 validation: PASSED")
else:
    print("02.29 Step 5 validation: NEEDS REVIEW")

02.29 - STEP 5 - FINAL PRIMARY KEY VALIDATION

customers_master
Rows             : 10,000
Primary key      : customer_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : VALID

products_master
Rows             : 2,500
Primary key      : product_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : VALID

stores_master
Rows             : 60
Primary key      : store_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : VALID

salespersons_master
Rows             : 500
Primary key      : salesperson_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : VALID

promotions_master
Rows             : 72
Primary key      : promotion_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : VALID

targets
Rows             : 8,980
Primary key      : target_id
Missing keys     : 0
Blank keys       : 0
Duplicate keys   : 0
Status            : 

In [7]:
print("=" * 70)
print("02.29 - STEP 6 - SAVE PRIMARY KEY VALIDATION REPORT")
print("=" * 70)

report_rows = []

# Master datasets
for dataset, (filename, key_column) in datasets_to_check.items():

    file_path = BASE_DIR / filename
    df = pd.read_csv(file_path, dtype={key_column: str})

    missing = int(df[key_column].isna().sum())
    blank = int(df[key_column].astype(str).str.strip().eq("").sum())
    duplicates = int(df[key_column].duplicated().sum())

    report_rows.append({
        "dataset": dataset,
        "primary_key": key_column,
        "rows": len(df),
        "missing_keys": missing,
        "blank_keys": blank,
        "duplicate_keys": duplicates,
        "status": "VALID" if (
            missing == 0 and
            blank == 0 and
            duplicates == 0
        ) else "NEEDS REVIEW"
    })


# Sales
report_rows.append({
    "dataset": "sales",
    "primary_key": "sales_line_id",
    "rows": 2118200,
    "missing_keys": 0,
    "blank_keys": 0,
    "duplicate_keys": sales_duplicate_count,
    "status": "VALID" if sales_duplicate_count == 0 else "NEEDS REVIEW"
})


# Returns
report_rows.append({
    "dataset": "returns",
    "primary_key": "return_id",
    "rows": 118400,
    "missing_keys": 0,
    "blank_keys": 0,
    "duplicate_keys": return_duplicate_count,
    "status": "VALID" if return_duplicate_count == 0 else "NEEDS REVIEW"
})


# Save report
report_df = pd.DataFrame(report_rows)

output_file = BASE_DIR / "primary_key_validation_report.csv"

report_df.to_csv(output_file, index=False)

print("\nValidation report:")
print(report_df.to_string(index=False))

print("\nReport saved:")
print(output_file)

if (report_df["status"] == "VALID").all():
    print("\n02.29 Step 6 validation: PASSED")
else:
    print("\n02.29 Step 6 validation: NEEDS REVIEW")

02.29 - STEP 6 - SAVE PRIMARY KEY VALIDATION REPORT

Validation report:
            dataset    primary_key    rows  missing_keys  blank_keys  duplicate_keys status
   customers_master    customer_id   10000             0           0               0  VALID
    products_master     product_id    2500             0           0               0  VALID
      stores_master       store_id      60             0           0               0  VALID
salespersons_master salesperson_id     500             0           0               0  VALID
  promotions_master   promotion_id      72             0           0               0  VALID
            targets      target_id    8980             0           0               0  VALID
              sales  sales_line_id 2118200             0           0               0  VALID
            returns      return_id  118400             0           0               0  VALID

Report saved:
..\data\synthetic\primary_key_validation_report.csv

02.29 Step 6 validation: PASSED
